# NVIDIA NIM: AIPerf ISL/OSL sweeps with live Grafana monitoring

This customer-ready notebook benchmarks an already-running Nemotron 3 Super 120B NIM, starts Prometheus + Grafana, and runs AIPerf profiles while leaving NIM context-window and maximum-sequence settings untouched.

Copy `.env.example` to `.env` and configure the endpoint, local NIM-cache tokenizer path, observability addresses, and offered load before running. AIPerf captures client and benchmark-window server metrics; Prometheus/Grafana provide near-real-time monitoring.

## 1. Imports and repository location

Select the **Python (bench)** kernel.

In [ ]:
from __future__ import annotations
import json, os, re, shlex, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display

def find_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate/"compose.observability.yaml").exists(): return candidate
    raise FileNotFoundError("Start Jupyter inside this repository")

REPO_ROOT=find_root(); COMPOSE_FILE=REPO_ROOT/"compose.observability.yaml"
ENV_FILE=REPO_ROOT/".env"
if not ENV_FILE.exists(): raise FileNotFoundError("Copy .env.example to .env and configure it before running the notebook")
load_dotenv(ENV_FILE, override=False)
RESULTS_ROOT=REPO_ROOT/"benchmark_results"; RESULTS_ROOT.mkdir(exist_ok=True)
AIPERF_EXE=str(Path(sys.executable).with_name("aiperf"))
print("Python:",sys.executable); print("Repository:",REPO_ROOT); print("Environment:",ENV_FILE); print("Results:",RESULTS_ROOT)

## 2. Customer environment and workload configuration

The model remains pinned to Nemotron 3 Super 120B. Values are loaded from the untracked repository-level `.env`; `.env.example` documents every setting.

Start the configured local tunnel from the repository root:

```bash
set -a; source .env; set +a
kubectl -n "$K8S_NAMESPACE" port-forward --address "$NIM_BIND_ADDRESS" service/"$K8S_NIM_SERVICE" "${NIM_LOCAL_PORT}:8000"
```

`NIM_PROMETHEUS_TARGET=host.docker.internal:18000` lets Prometheus scrape through the same tunnel. This requires `NIM_BIND_ADDRESS=0.0.0.0`; restrict the port with a host firewall. When Docker can reach the Kubernetes service directly, use that service address and bind only to `127.0.0.1`. The AIPerf concurrency and ISL/OSL values are client-side offered load; this notebook does not set `NIM_MAX_MODEL_LEN`, `--max-model-len`, `NIM_MAX_BATCH_SIZE`, or `--max-num-seqs`.

In [ ]:
def env(name, default=None, required=False):
    value=os.getenv(name, default)
    if required and not value: raise ValueError(f"Missing required setting: {name}")
    return value

def env_int(name, default): return int(env(name,str(default)))
def env_bool(name, default=False): return str(env(name,str(default))).strip().lower() in {"1","true","yes","on"}
def env_int_list(name, default): return [int(x.strip()) for x in env(name,default).split(",") if x.strip()]

MODEL_ID=env("NIM_MODEL_ID","nvidia/nemotron-3-super-120b-a12b")
TOKENIZER_PATH=env("NIM_TOKENIZER_PATH",required=True)
MODELS=[{"name":MODEL_ID,"tokenizer":TOKENIZER_PATH}]
NIM_BASE_URL=env("NIM_BASE_URL","http://127.0.0.1:18000")
NIM_METRICS_URL=env("NIM_METRICS_URL",NIM_BASE_URL.rstrip("/")+"/v1/metrics")
NIM_PROMETHEUS_TARGET=env("NIM_PROMETHEUS_TARGET","host.docker.internal:18000")
NIM_API_KEY=env("NIM_API_KEY","")
PROMETHEUS_URL=env("PROMETHEUS_URL","http://localhost:9090").rstrip("/")
GRAFANA_URL=env("GRAFANA_URL","http://localhost:3001").rstrip("/")
GRAFANA_PUBLIC_URL=env("GRAFANA_PUBLIC_URL",GRAFANA_URL).rstrip("/")
GRAFANA_AUTH=(env("GRAFANA_USER","admin"),env("GRAFANA_PASSWORD","admin"))
OBSERVABILITY_MANAGED_EXTERNALLY=env_bool("OBSERVABILITY_MANAGED_EXTERNALLY",False)

ISL_VALUES=env_int_list("AIPERF_ISL_VALUES","128,512,1024")
OSL_VALUES=env_int_list("AIPERF_OSL_VALUES","32,128,256,512")
SWEEP_TYPE=env("AIPERF_SWEEP_TYPE","grid")
CONCURRENCY=env_int("AIPERF_CONCURRENCY",32)
REQUEST_COUNT=env_int("AIPERF_REQUEST_COUNT",64)
WARMUP_REQUEST_COUNT=env_int("AIPERF_WARMUP_REQUEST_COUNT",16)
SMOKE_REQUEST_COUNT=env_int("AIPERF_SMOKE_REQUEST_COUNT",32)
SMOKE_WARMUP_REQUEST_COUNT=env_int("AIPERF_SMOKE_WARMUP_REQUEST_COUNT",1)
NUM_DATASET_ENTRIES=env_int("AIPERF_NUM_DATASET_ENTRIES",200)
RANDOM_SEED=env_int("AIPERF_RANDOM_SEED",17)
FORCE_EXACT_OSL=env_bool("AIPERF_FORCE_EXACT_OSL",True)
SWEEP_COOLDOWN_SECONDS=env_int("AIPERF_SWEEP_COOLDOWN_SECONDS",10)
NUM_PROFILE_RUNS=env_int("AIPERF_NUM_PROFILE_RUNS",1)
PROFILE_RUN_COOLDOWN_SECONDS=env_int("AIPERF_PROFILE_RUN_COOLDOWN_SECONDS",10)
RUN_SMOKE_TEST=env_bool("RUN_SMOKE_TEST",True)
RUN_FULL_SWEEP=env_bool("RUN_FULL_SWEEP",False)

assert MODEL_ID=="nvidia/nemotron-3-super-120b-a12b", "This demo is specific to Nemotron 3 Super 120B"
assert Path(TOKENIZER_PATH).is_dir(), f"Tokenizer path not found: {TOKENIZER_PATH}"
assert SWEEP_TYPE in {"grid","zip"}
if SWEEP_TYPE=="zip": assert len(ISL_VALUES)==len(OSL_VALUES)
assert 1 <= NUM_PROFILE_RUNS <= 10
case_count=len(ISL_VALUES)*len(OSL_VALUES) if SWEEP_TYPE=="grid" else len(ISL_VALUES)
print(f"Variations: {case_count}; concurrency: {CONCURRENCY}; trials/variation: {NUM_PROFILE_RUNS}")
print("Server sizing: runtime defaults (not specified by this notebook)")

## 3. Preflight AIPerf and NIM

This read-only check validates AIPerf, the configured local-cache tokenizer, the exact Nemotron model ID, `/v1/models`, and `/v1/metrics`. The model response also exposes the context length resolved by the running NIM.

In [ ]:
def api_headers(): return {"Authorization":f"Bearer {NIM_API_KEY}"} if NIM_API_KEY else {}
def get_json(path):
    x=requests.get(NIM_BASE_URL.rstrip("/")+path,headers=api_headers(),timeout=10); x.raise_for_status(); return x.json()

checks=[]
required_executables=(AIPERF_EXE,) if OBSERVABILITY_MANAGED_EXTERNALLY else (AIPERF_EXE,"docker")
for exe in required_executables:
    found=exe if Path(exe).is_file() else shutil.which(exe); checks.append({"check":Path(exe).name,"ok":bool(found),"detail":found or "not found"})
try:
    v=subprocess.run([AIPERF_EXE,"--version"],check=True,capture_output=True,text=True).stdout.strip()
    checks.append({"check":"AIPerf version","ok":True,"detail":v})
except Exception as e: checks.append({"check":"AIPerf version","ok":False,"detail":str(e)})
discovered=[]
try:
    discovered=[x["id"] for x in get_json("/v1/models").get("data",[])]
    checks.append({"check":"NIM /v1/models","ok":bool(discovered),"detail":", ".join(discovered)})
except Exception as e: checks.append({"check":"NIM /v1/models","ok":False,"detail":str(e)})
try:
    x=requests.get(NIM_METRICS_URL,headers=api_headers(),timeout=10); x.raise_for_status()
    count=sum(bool(line and not line.startswith("#")) for line in x.text.splitlines())
    checks.append({"check":"NIM /v1/metrics","ok":count>0,"detail":f"{count:,} series"})
except Exception as e: checks.append({"check":"NIM /v1/metrics","ok":False,"detail":str(e)})
checks.append({"check":"tokenizer cache","ok":Path(TOKENIZER_PATH).is_dir(),"detail":TOKENIZER_PATH})
checks.append({"check":"configured model","ok":MODEL_ID in discovered,"detail":MODEL_ID})
display(pd.DataFrame(checks))
if discovered: print("Discovered model ID(s):",discovered)

## 4. Start Prometheus and Grafana

Prometheus scrapes NIM every two seconds; Grafana refreshes every five seconds.

In [ ]:
target=REPO_ROOT/"observability/prometheus/targets/nim.json"
target.write_text(json.dumps([{"targets":[NIM_PROMETHEUS_TARGET],"labels":{"service":"nim-llm"}}],indent=2)+"\n")
if OBSERVABILITY_MANAGED_EXTERNALLY:
    print("Observability is managed by the outer Compose deployment")
else:
    cmd=["docker","compose","--env-file",str(ENV_FILE),"-f",str(COMPOSE_FILE),"up","-d"]
    print("$",shlex.join(cmd)); subprocess.run(cmd,cwd=REPO_ROOT,check=True)
for url in (PROMETHEUS_URL+"/-/ready",GRAFANA_URL+"/api/health"):
    for _ in range(30):
        try:
            if requests.get(url,timeout=2).ok: print("Ready:",url); break
        except requests.RequestException: pass
        time.sleep(1)
    else: raise RuntimeError("Not ready: "+url)
display(Markdown(f"[Open the NIM live dashboard]({GRAFANA_PUBLIC_URL}/d/nim-llm-overview?refresh=5s&from=now-15m&to=now)"))

### Confirm Prometheus can scrape NIM

`up` must equal `1`; this catches container-to-host routing problems.

In [ ]:
time.sleep(3)
x=requests.get(PROMETHEUS_URL+"/api/v1/query",params={"query":'up{job="nim-llm"}'},timeout=10); x.raise_for_status()
scrapes=x.json()["data"]["result"]
display(pd.DataFrame([{"instance":i["metric"].get("instance"),"up":float(i["value"][1])} for i in scrapes]))
assert scrapes and all(float(i["value"][1])==1 for i in scrapes), "Check NIM_PROMETHEUS_TARGET and the configured Prometheus targets page"

## 5. Build and preview native AIPerf commands

Comma-separated `--isl` and `--osl` values become a native sweep. `grid` runs the Cartesian product; `zip` pairs values element-wise. AIPerf also scrapes NIM `/v1/metrics` during each benchmark and writes JSON/CSV/JSONL/Parquet server-metric artifacts.

When `FORCE_EXACT_OSL=True`, `ignore_eos:true` makes compatible NIM/vLLM backends continue to the requested limit. Disable it to measure natural stopping behavior.

In [ ]:
def slug(x): return re.sub(r"[^A-Za-z0-9_.-]+","-",x).strip("-")[:100]
def common_cmd(model,out,request_count=REQUEST_COUNT,warmup_request_count=WARMUP_REQUEST_COUNT):
    cmd=[AIPERF_EXE,"profile","--model",model["name"],"--tokenizer",model["tokenizer"],
      "--endpoint-type","chat","--url",NIM_BASE_URL,"--streaming","--concurrency",str(CONCURRENCY),
      "--request-count",str(request_count),"--warmup-request-count",str(warmup_request_count),
      "--num-dataset-entries",str(NUM_DATASET_ENTRIES),"--random-seed",str(RANDOM_SEED),
      "--server-metrics",NIM_METRICS_URL,"--server-metrics-formats","json","csv","jsonl","parquet",
      "--artifact-dir",str(out),"--export-level","records","--ui","none"]
    if FORCE_EXACT_OSL: cmd += ["--extra-inputs","ignore_eos:true"]
    if NIM_API_KEY: cmd += ["--api-key",NIM_API_KEY]
    return cmd
def smoke_cmd(model,out): return common_cmd(model,out,SMOKE_REQUEST_COUNT,SMOKE_WARMUP_REQUEST_COUNT)+["--isl",str(min(ISL_VALUES)),"--osl",str(min(OSL_VALUES))]
def sweep_cmd(model,out):
    return common_cmd(model,out)+["--isl",",".join(map(str,ISL_VALUES)),"--osl",",".join(map(str,OSL_VALUES)),
      "--sweep-type",SWEEP_TYPE,"--parameter-sweep-same-seed",
      "--parameter-sweep-cooldown-seconds",str(SWEEP_COOLDOWN_SECONDS),
      "--num-profile-runs",str(NUM_PROFILE_RUNS),"--profile-run-cooldown-seconds",str(PROFILE_RUN_COOLDOWN_SECONDS)]
def redact(cmd):
    x=cmd.copy()
    for i,v in enumerate(x[:-1]):
        if v=="--api-key": x[i+1]="***"
    return shlex.join(x)

preview=RESULTS_ROOT/"preview"/slug(MODELS[0]["name"])
print("SMOKE:")
print(redact(smoke_cmd(MODELS[0],preview/"smoke")))
print()
print("NATIVE SWEEP:")
print(redact(sweep_cmd(MODELS[0],preview/"sweep")))

## 6. Execution helper and smoke test

Output streams to the notebook and `aiperf-driver.log`. Grafana gets start/end annotations for the smoke profile or model sweep. Native variation names remain visible in AIPerf's artifact tree and sweep aggregate.

In [ ]:
def annotation(text,tags):
    try:
        x=requests.post(GRAFANA_URL+"/api/annotations",auth=GRAFANA_AUTH,
          json={"time":int(time.time()*1000),"tags":["aiperf",*tags],"text":text},timeout=5); x.raise_for_status()
    except requests.RequestException as e: print("Annotation warning:",e)

def run_command(cmd,out,label,tags):
    out.mkdir(parents=True,exist_ok=True); start=datetime.now(timezone.utc); annotation("START "+label,tags)
    print("\n"+"="*80); print(start.isoformat(),label); print("$",redact(cmd),flush=True)
    log_path=out/"aiperf-driver.log"
    with log_path.open("w",encoding="utf-8") as log:
        proc=subprocess.Popen(cmd,cwd=REPO_ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout: print(line,end=""); log.write(line)
        rc=proc.wait()
    end=datetime.now(timezone.utc); annotation(f"END ({'ok' if rc==0 else 'failed'}) "+label,tags)
    result={"label":label,"started_at_utc":start.isoformat(),"ended_at_utc":end.isoformat(),
      "duration_seconds":(end-start).total_seconds(),"return_code":rc,"artifact_dir":str(out),"driver_log":str(log_path)}
    if rc: raise RuntimeError(f"AIPerf failed; see {log_path}")
    return result

if RUN_SMOKE_TEST:
    assert not any(m["name"].startswith("REPLACE_") for m in MODELS),"Select a model first"
    model=MODELS[0]; out=RESULTS_ROOT/f"smoke-{datetime.now().strftime('%Y%m%d-%H%M%S')}"/slug(model["name"])
    display(pd.DataFrame([run_command(smoke_cmd(model,out),out,f"SMOKE {model['name']} ISL={min(ISL_VALUES)} OSL={min(OSL_VALUES)}",[slug(model['name']),"smoke"])]))
else: print("Skipped. Set RUN_SMOKE_TEST=True to run the Nemotron smoke profile.")

## 7. Run native ISL × OSL sweeps

One AIPerf command handles every variation for a model, including cooldowns, repeats, confidence reporting, failed-cell tracking, and sweep aggregate generation.

In [ ]:
manifest=[]
if RUN_FULL_SWEEP:
    assert not any(m["name"].startswith("REPLACE_") for m in MODELS),"Select model(s) first"
    root=RESULTS_ROOT/f"sweep-{datetime.now().strftime('%Y%m%d-%H%M%S')}"; root.mkdir(parents=True,exist_ok=False)
    for number,model in enumerate(MODELS,1):
        model_slug=slug(model["name"]); out=root/model_slug
        label=f"AIPerf {SWEEP_TYPE} {model['name']} ({len(ISL_VALUES)} ISL × {len(OSL_VALUES)} OSL)"
        print(f"\nModel {number}/{len(MODELS)}")
        row={"model":model["name"],"tokenizer":model["tokenizer"],"sweep_type":SWEEP_TYPE,
          "isl_values":json.dumps(ISL_VALUES),"osl_values":json.dumps(OSL_VALUES),"profile_runs":NUM_PROFILE_RUNS}
        row.update(run_command(sweep_cmd(model,out),out,label,[model_slug,"sweep",SWEEP_TYPE]))
        manifest.append(row); pd.DataFrame(manifest).to_csv(root/"manifest.csv",index=False)
    print("Sweep complete:",root); display(pd.DataFrame(manifest))
else: print("Skipped. Run smoke first, then set RUN_FULL_SWEEP=True.")

## 8. Index and inspect AIPerf results

Sweep aggregates are stable programmatic inputs containing parameters, per-combination metrics, failed runs, confidence statistics, and Pareto-optimal cells. Server metric files contain benchmark-window NIM/vLLM telemetry.

In [ ]:
summary_files=sorted(RESULTS_ROOT.rglob("profile_export_aiperf*.json"))
sweep_files=sorted(RESULTS_ROOT.rglob("*sweep*.json"))
server_files=sorted(RESULTS_ROOT.rglob("*server_metrics*.json"))
index=pd.DataFrame({"kind":(["client summary"]*len(summary_files)+["sweep aggregate"]*len(sweep_files)+["server metrics"]*len(server_files)),
 "path":[str(p.relative_to(REPO_ROOT)) for p in summary_files+sweep_files+server_files]})
display(index.drop_duplicates())
if sweep_files:
    latest=sweep_files[-1]; data=json.loads(latest.read_text())
    print("Latest sweep aggregate:",latest)
    combos=data.get("per_combination_metrics",[])
    rows=[]
    for combo in combos:
        row={"artifact":str(latest.relative_to(REPO_ROOT)),**combo.get("parameters",{})}
        for name,value in combo.get("metrics",{}).items():
            if isinstance(value,dict) and "mean" in value: row[name]=value["mean"]
        rows.append(row)
    display(pd.DataFrame(rows) if rows else JSON(data,expanded=False))
elif summary_files:
    latest=summary_files[-1]; print("Latest summary:",latest); display(JSON(json.loads(latest.read_text()),expanded=False))
else: print("No AIPerf results yet")

## 9. Stop observability (optional)

This preserves volumes. Use `down -v` only when intentionally deleting retained data.

In [ ]:
STOP_OBSERVABILITY=False
if STOP_OBSERVABILITY and OBSERVABILITY_MANAGED_EXTERNALLY:
    print("Stop the outer Compose deployment from the host")
elif STOP_OBSERVABILITY:
    subprocess.run(["docker","compose","--env-file",str(ENV_FILE),"-f",str(COMPOSE_FILE),"down"],cwd=REPO_ROOT,check=True)
else: print("Observability stack left running")